#### 📋 The Execution Timeline Audit

Because you are using a continuous daily calendar (`freq='D'`), your code perfectly bridges the gap between weekend publication data and weekday market realities. Here is exactly how a signal moves through your pipeline:

| Timeline Point | Data / Signal State | Code Vector Mapping | Portfolio Action |
| --- | --- | --- | --- |
| **Day $t$ (7:00 AM)** | *People's Daily* is scraped; Ad Share of Voice (SoV) is updated. | `signal_df` at index $t$ captures today's raw trading signal ($1$, $0$, or $-1$). | Signal calculated before market open. |
| **Day $t$ (9:30 AM - 3:00 PM)** | Market trades. Your portfolio maintains its existing tracking positions. | `position_df` at index $t$ uses `ffill()` to sustain or alter your asset state. | No immediate execution changes yet. |
| **Day $t$ (3:00 PM Close)** | Market Closes. You execute rebalancing trades based on Day $t$'s morning data. | `signal_weight` at index $t+1$ is assigned the allocation derived from `position_df[t]`. | **Trade Execution** occurs here at Day $t$ closing prices. |
| **Day $t+1$ (Full Day)** | Market executes next session's price returns. | `portfolio_gross_roi[t+1]` = `signal_weight[t+1]` $\times$ `industry_roi[t+1]`. | Your capital captures the pure Close-to-Close return of Day $t+1$. |

#### The Weekend Continuity Benefit

Because *People's Daily* publishes seven days a week, a signal generated on Saturday or Sunday will forward-fill seamlessly. Your `.shift(1)` on a daily frequency ensures that a Sunday morning signal naturally applies its portfolio weights to Monday's market session, exactly when the liquidity pools open.

#### 🔍 Fine-Tuning for Real-World Deployment

While the vectorization logic is pristine, you should account for three practical nuances before using these results for capital allocation:

#### 1. The Multi-Asset "Weight Drift" Approximation

In `Position2Weight`, you compute equal weights based on the target position vector:

```python
weights = position_df.div(num_stocks_held, axis=0).fillna(0.0)

```

In a pure matrix backtest, multiplying this weight matrix directly by your daily ROI matrix assumes that your portfolio rebalances back to *exact equal weights* at the close of **every single day**.

* **The Reality:** If you hold 3 sector ETFs and one jumps 5% today while the others drop, their natural weights drift.
* **Impact:** For daily signals, this difference is negligible. However, if your signals remain unchanged for weeks at a time, your backtest will slightly miscalculate the compounding returns by forcing a daily rebalance that you might not actually execute in your broker account.

#### 2. Turnover Fee Sizing

Your transaction fee logic is highly elegant:

```python
weight_changes = weight_df.diff().fillna(0.0)
fees_df = (weight_changes.clip(lower=0) * buy_fee) + (weight_changes.clip(upper=0).abs() * sell_fee)

```

> **Note:** Because your `weight_df` includes the `.shift(1)` calculation, the `diff()` function correctly aligns transaction fees to the exact calendar day the trade is executed. When transitioning from an active holding state to a cash state (all zeros), your turnover will accurately capture the entire liquidating sell fee.

#### 3. Surviving Matrix Gaps

Ensure that your underlying asset returns file (`ETF_DATA_PATH + file_name`) utilizes **Split and Dividend Adjusted Close Prices** (Total Return Index data). Using unadjusted market close prices will cause artificial drawdowns in your `cumulative_portfolio_return` whenever an industry index or sector ETF distributes dividends or undergoes structural splits.

---

#### The Memory Leak in `Signal2Position`

Your custom conditional logic inside `Signal2Position` contains a structural bug. You wrote rules to maintain a position when a signal drops to `0.0`:

```python
positions[(signals_shifted == 1.0) & (signal_df == 0.0)] = 1.0  # Hold

```

The Problem: `signals_shifted` is defined as `signal_df.shift(1)`. It looks at the *raw signals*, not your *active portfolio positions*.

If your signal timeline looks like this over four consecutive days:

* **Day 1:** `0`
* **Day 2:** `1` $\rightarrow$ (Shifted=0, Signal=1) $\rightarrow$ Position becomes `1` (Correct)
* **Day 3:** `0` $\rightarrow$ (Shifted=1, Signal=0) $\rightarrow$ Position becomes `1` (Correctly held)
* **Day 4:** `0` $\rightarrow$ (Shifted=0, Signal=0) $\rightarrow$ Position drops to `0` **(Bug!)**

On Day 4, even though you were holding the asset on Day 3, `signals_shifted` (which is yesterday's raw signal) evaluates to `0`. Your code falls into the `(signals_shifted == 0.0) & (signal_df == 0.0) = 0.0` rule, causing you to accidentally exit your position. Your "hold" rule only lasts for exactly one day before forcing an exit.

* **`ffill()` State Machine:** Replacing `0` with `NaN` and using `.ffill()` tracks long positions seamlessly across neutral days, entirely bypassing your nested dictionary masks.

In [ ]:
def Signal2Position(signal_df, frequency=1):
    """
    We change our position daily (default). Now, we get position dataframe (0 or 1) based on the signals.
    (Explanation: position = 1 means we hold the stock, position = 0 means we do not hold the stock)
    (If the instrument can be shorted, position = -1 means we short the stock)
    (Note that we suppose the instruments cannot be shorted in this example.)

    In order to avoid future data leakage, we will shift the signals by one day when calculating weights.
    (i.e. the weight on day t is based on the signal generated on day t-1)

    In the position changing day, we will take the full position according to the signal.
    There are 9 possible situations:
    Previous signal = 0.0, Current signal = 1.0  => position = 1.0 (buy)
    Previous signal = 0.0, Current signal = -1.0 => position = 0.0 (do nothing, cannot short)
    Previous signal = 0.0, Current signal = 0.0  => position = 0.0 (do nothing)
    Previous signal = 1.0, Current signal = 1.0  => position = 1.0 (hold)
    Previous signal = 1.0, Current signal = -1.0 => position = 0.0 (sell)
    Previous signal = 1.0, Current signal = 0.0  => position = 1.0 (hold)
    Previous signal = -1.0, Current signal = 1.0 => position = 1.0 (buy)
    Previous signal = -1.0, Current signal = -1.0 => position = 0.0 (do nothing, cannot short)
    Previous signal = -1.0, Current signal = 0.0 => position = 0.0 (do nothing, cannot short)

    More generally, if we change the position weekly (or other frequency), we can follow the same logic.
    First, add a new column 'week' to the signals DataFrame, which indicates the week number of each date.
    Then, we keep only the signals on each position changing day (e.g., Monday) (After the signals are shifted).
    Then, we can follow the same logic as above to calculate the positions.
    Finally, we reindex the positions DataFrame to match the original signals DataFrame, forward filling the positions for the days in between.
    """
    signals_shifted = signal_df.shift(frequency).fillna(0.0)
    # We change our position daily. Now, we get position dataframe (0 or 1) based on the signals.

    positions = signals_shifted.copy()
    positions[(signals_shifted == 0.0) & (signal_df == 1.0)] = 1.0  # buy
    positions[(signals_shifted == 0.0) & (signal_df == -1.0)] = 0.0  # do nothing, cannot short
    positions[(signals_shifted == 0.0) & (signal_df == 0.0)] = 0.0  # do nothing
    positions[(signals_shifted == 1.0) & (signal_df == 1.0)] = 1.0  # hold
    positions[(signals_shifted == 1.0) & (signal_df == -1.0)] = 0.0  # sell
    positions[(signals_shifted == 1.0) & (signal_df == 0.0)] = 1.0  # hold
    positions[(signals_shifted == -1.0) & (signal_df == 1.0)] = 1.0  # buy
    positions[(signals_shifted == -1.0) & (signal_df == -1.0)] = 0.0  # do nothing, cannot short
    positions[(signals_shifted == -1.0) & (signal_df == 0.0)] = 0.0  # do nothing, cannot short
    
    return positions

# Fixed

def Signal2Position(signal_df):
    """
    🚨CRUCIAL WARNING: People's daily is released at the early morning each day (day t) and it is before traing hour (9:30 AM).
    Therefore, the signal generated on day t is based on the data of day t (today).

    We change our position daily (default). Now, we get position dataframe (0 or 1) based on the signals.
    (Explanation: position = 1 means we hold the stock, position = 0 means we do not hold the stock)
    (If the instrument can be shorted, position = -1 means we short the stock)
    (Note that we suppose the instruments cannot be shorted in this example.)
    """
    # We fill neutral (0) positions with the previous day's state to properly track "Hold"
    position_df = signal_df.replace(0.0, np.nan).ffill().fillna(0.0)
    position_df[position_df == -1.0] = 0.0  # Long-only constraint (converts short triggers to cash)
    return position_df

---

Your original code works correctly, but it has severe performance bottlenecks that will slow it down dramatically as your dataset grows.

Core Bottlenecks in Your Original Code

* **The I/O Overhead (Double Reading):** You loop through `SW_SECTOR_PATH_MAPPING` and `BENCHMARK_PATH_MAPPING` **twice** (once for PnL and once for ROI). This means every single CSV file is being opened, parsed, and loaded into memory two separate times.
* **The `pd.merge` Loop Penalty:** Using `pd.merge(..., how="inner")` inside a sequential loop is computationally expensive. Pandas is forced to recreate a new DataFrame object, re-allocate memory, and re-index the rows on every iteration.
* **String-Based Date Filtering:** Filtering date sequences using string lists (`.isin(date_range_str)`) drops the performance advantages of built-in Pandas datetime engines.

The Optimized Solution

We can compress your entire calculation pipeline by unifying the logic into a single helper function that reads each file **exactly once**, extracts both metrics, and constructs the final DataFrames instantly using dictionary mapping.

Key Improvements Explained:

* **Dictionary Pooling:** Instead of merging columns one by one via `pd.merge`, we save each raw pandas Series into a collection dictionary (`ind_roi_pool`). Passing a dictionary of Series directly into `pd.DataFrame(dictionary, index=...)` allows pandas to construct and align the entire layout in memory in one highly optimized step.
* **`parse_dates=['date']`:** Instructs the CSV reader to interpret dates as binary timestamps upon loading. This accelerates the subsequent relational logic (`>=` and `<=`) exponentially compared to matching raw string characters.
* **Preserved Output Formats:** The resulting DataFrames fully preserve your layout, keeping chronological datetime objects as the index, named `"date"`, with industries/benchmarks mapped as the columns.

In [ ]:
SW_SECTOR_PATH_MAPPING = JsonFile_to_Dict(ETF_DATA_PATH + "SW-SECTOR-PATH-MAPPING.json")
BENCHMARK_PATH_MAPPING = JsonFile_to_Dict(ETF_DATA_PATH + "BENCHMARK-PATH-MAPPING.json")
SW_SECTOR_LEVEL_1_List = [industry for industry in SW_SECTOR_PATH_MAPPING.keys() if len(industry.split('-')) == 1]
BENCHMARK_List = [benchmark for benchmark in BENCHMARK_PATH_MAPPING.keys()]

def Get_Index_ROI(industry_name, sw_sector_path_map, begin_date, end_date):
    SW_Real_Estate_DF = pd.read_csv(ETF_DATA_PATH + sw_sector_path_map[industry_name])
    # Note the date format in the CSV file is YYYY-MM-DD
    date_range = pd.date_range(start=begin_date, end=end_date)
    date_range_str = date_range.strftime("%Y-%m-%d").tolist()
    SW_Real_Estate_DF_Filtered = SW_Real_Estate_DF[SW_Real_Estate_DF['date'].isin(date_range_str)].reset_index(drop=True)[['date', 'open', 'close']]
    
    # ROI Calculation
    # For the first day, ROI = (close - open) / open
    # For subsequent days, ROI = (current_close - previous_close) / previous_close
    SW_Real_Estate_DF_Filtered_ROI_DF = SW_Real_Estate_DF_Filtered.copy()
    First_day_ROI = (SW_Real_Estate_DF_Filtered_ROI_DF.loc[0, 'close'] - SW_Real_Estate_DF_Filtered_ROI_DF.loc[0, 'open']) / SW_Real_Estate_DF_Filtered_ROI_DF.loc[0, 'open']
    SW_Real_Estate_DF_Filtered_ROI_DF['ROI'] = SW_Real_Estate_DF_Filtered_ROI_DF['close'].pct_change().fillna(First_day_ROI)
    SW_Real_Estate_DF_Filtered_ROI_DF = SW_Real_Estate_DF_Filtered_ROI_DF[['date', 'ROI']]
    
    # Ensure every date in the range has data
    # If missing, fill with 0 ROI
    SW_Real_Estate_DF_Filtered_ROI_DF.set_index('date', inplace=True)
    SW_Real_Estate_DF_Filtered_ROI = SW_Real_Estate_DF_Filtered_ROI_DF.reindex(date_range_str, fill_value=0).reset_index().rename(columns={'index': 'date'})
    SW_Real_Estate_DF_Filtered_ROI["PnL"] = (1 + SW_Real_Estate_DF_Filtered_ROI['ROI']).cumprod()
    
    # Get the independent PnL series (suppose begin with 1 unit of currency)
    # SW_Real_Estate_PnL_List = list((1 + SW_Real_Estate_DF_Filtered_ROI['ROI']).cumprod())
    return SW_Real_Estate_DF_Filtered_ROI

def Get_Benchmark_ROI(benchmark_name, benchmark_path_map, begin_date, end_date):
    Benchmark_DF = pd.read_csv(ETF_DATA_PATH + benchmark_path_map[benchmark_name])
    # Note the date format in the CSV file is YYYY-MM-DD
    date_range = pd.date_range(start=begin_date, end=end_date)
    date_range_str = date_range.strftime("%Y-%m-%d").tolist()
    Benchmark_DF_Filtered = Benchmark_DF[Benchmark_DF['date'].isin(date_range_str)].reset_index(drop=True)[['date', 'open', 'close']]
    
    # ROI Calculation
    # For the first day, ROI = (close - open) / open
    # For subsequent days, ROI = (current_close - previous_close) / previous_close
    Benchmark_DF_Filtered_ROI_DF = Benchmark_DF_Filtered.copy()
    First_day_ROI = (Benchmark_DF_Filtered_ROI_DF.loc[0, 'close'] - Benchmark_DF_Filtered_ROI_DF.loc[0, 'open']) / Benchmark_DF_Filtered_ROI_DF.loc[0, 'open']
    Benchmark_DF_Filtered_ROI_DF['ROI'] = Benchmark_DF_Filtered_ROI_DF['close'].pct_change().fillna(First_day_ROI)
    Benchmark_DF_Filtered_ROI_DF = Benchmark_DF_Filtered_ROI_DF[['date', 'ROI']]
    
    # Ensure every date in the range has data
    # If missing, fill with 0 ROI
    Benchmark_DF_Filtered_ROI_DF.set_index('date', inplace=True)
    Benchmark_DF_Filtered_ROI = Benchmark_DF_Filtered_ROI_DF.reindex(date_range_str, fill_value=0).reset_index().rename(columns={'index': 'date'})
    Benchmark_DF_Filtered_ROI["PnL"] = (1 + Benchmark_DF_Filtered_ROI['ROI']).cumprod()
    
    # Get the independent PnL series (suppose begin with 1 unit of currency)
    # Benchmark_PnL_List = list((1 + Benchmark_DF_Filtered_ROI['ROI']).cumprod())
    return Benchmark_DF_Filtered_ROI

# Get all level 1 industry PnL data
industry_level1_pnl_all_df = pd.DataFrame()
for industry in SW_SECTOR_PATH_MAPPING:
    if len(industry.split("-")) == 1: # get 1 level industry
        industry_roi_pnl_df = Get_Index_ROI(
            industry_name=industry, sw_sector_path_map=SW_SECTOR_PATH_MAPPING, 
            begin_date=Begin_date, end_date=End_date)
        industry_pnl_df = industry_roi_pnl_df[["date", "PnL"]].rename(columns={"PnL": industry})
        # Combine dfs
        if industry_level1_pnl_all_df.empty:
            industry_level1_pnl_all_df = industry_pnl_df
        else:
            industry_level1_pnl_all_df = pd.merge(industry_level1_pnl_all_df, industry_pnl_df, on="date", how="inner")

industry_level1_pnl_all_df["date"] = pd.to_datetime(industry_level1_pnl_all_df["date"])

# Get all level 1 industry ROI data
industry_level1_roi_all_df = pd.DataFrame()
for industry in SW_SECTOR_PATH_MAPPING:
    if len(industry.split("-")) == 1: # get 1 level industry
        industry_roi_pnl_df = Get_Index_ROI(
            industry_name=industry, sw_sector_path_map=SW_SECTOR_PATH_MAPPING, 
            begin_date=Begin_date, end_date=End_date)
        industry_roi_df = industry_roi_pnl_df[["date", "ROI"]].rename(columns={"ROI": industry})
        # Combine dfs
        if industry_level1_roi_all_df.empty:
            industry_level1_roi_all_df = industry_roi_df
        else:
            industry_level1_roi_all_df = pd.merge(industry_level1_roi_all_df, industry_roi_df, on="date", how="inner")

industry_level1_roi_all_df["date"] = pd.to_datetime(industry_level1_roi_all_df["date"])

# Get all benchmark PnL data
benchmark_pnl_all_df = pd.DataFrame()
for benchmark in BENCHMARK_PATH_MAPPING:
    benchmark_roi_pnl_df = Get_Benchmark_ROI(
        benchmark_name=benchmark, benchmark_path_map=BENCHMARK_PATH_MAPPING, 
        begin_date=Begin_date, end_date=End_date)
    benchmark_pnl_df = benchmark_roi_pnl_df[["date", "PnL"]].rename(columns={"PnL": benchmark})
    # Combine dfs
    if benchmark_pnl_all_df.empty:
        benchmark_pnl_all_df = benchmark_pnl_df
    else:
        benchmark_pnl_all_df = pd.merge(benchmark_pnl_all_df, benchmark_pnl_df, on="date", how="inner")

benchmark_pnl_all_df["date"] = pd.to_datetime(benchmark_pnl_all_df["date"])

# Get all benchmark ROI data
benchmark_roi_all_df = pd.DataFrame()
for benchmark in BENCHMARK_PATH_MAPPING:
    benchmark_roi_pnl_df = Get_Benchmark_ROI(
        benchmark_name=benchmark, benchmark_path_map=BENCHMARK_PATH_MAPPING, 
        begin_date=Begin_date, end_date=End_date)
    benchmark_roi_df = benchmark_roi_pnl_df[["date", "ROI"]].rename(columns={"ROI": benchmark})
    # Combine dfs
    if benchmark_roi_all_df.empty:
        benchmark_roi_all_df = benchmark_roi_df
    else:
        benchmark_roi_all_df = pd.merge(benchmark_roi_all_df, benchmark_roi_df, on="date", how="inner")

benchmark_roi_all_df["date"] = pd.to_datetime(benchmark_roi_all_df["date"])